In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

csv_path = Path("../data/processed/df_full.csv")
parquet_path = Path("../data/processed/df_full.parquet")

if not parquet_path.exists():
    print("CSV conversion → Parquet")
    df = pd.read_csv(csv_path)
    df.to_parquet(parquet_path, engine="fastparquet")
else:
    print("Parquet already present, conversion ignored")

In [ ]:
df = pd.read_parquet("../data/processed/df_full.parquet", engine="fastparquet")

# **STEP 1**: CREATING THE USER PROFILE (USER LEVEL)

In [ ]:
# We need to know how many items there are per order for the volume
# We first group by order to get the basket size
order_sizes = df.groupby(['user_id', 'order_id']).size().reset_index(name='basket_size')

In [ ]:
# We create a view of the data WITHOUT the first commands so as not to be polluted by the NaNs from the first time
df_loyalty = df[df['order_number'] > 1]

user_profiles = df_loyalty.groupby('user_id').agg({'days_since_prior_order': 'mean'}).reset_index()

# For users who have placed ONLY ONE order in total (and therefore do not appear in df_loyalty), they are assigned the maximum value (e.g., 30 days) to classify them as “LowFreq.”
all_users = pd.DataFrame(df['user_id'].unique(), columns=['user_id'])
user_profiles = all_users.merge(user_profiles, on='user_id', how='left')
user_profiles['days_since_prior_order'] = user_profiles['days_since_prior_order'].fillna(30)

avg_basket = order_sizes.groupby('user_id')['basket_size'].mean().reset_index()
user_profiles = user_profiles.merge(avg_basket, on='user_id')

In [ ]:
# Group by order_id to have the list of products bought together
transactions = df.groupby(['user_id', 'order_id'])['product_name'].apply(list).reset_index()

# **STEP 2**: PERSONAS DEFINITION (STRATIFICATION)

In [ ]:
# We create categories to “discretize” our users.
# Example: We divide into three tiers (Low, Medium, High) for frequency and volume.

#1. Frequency Category (0 = Buys often, 2 = Buys rarely)
user_profiles['freq_cat'] = pd.qcut(user_profiles['days_since_prior_order'], q=3, labels=['HighFreq', 'MedFreq', 'LowFreq'])

# 2. Volume Category (0 = Small basket size, 2 = Big Basket size)
user_profiles['vol_cat'] = pd.qcut(user_profiles['basket_size'], q=3, labels=['SmallBasket', 'MedBasket', 'BigBasket'])

# 3. Creation of the combined 'Strate' (ex: "HighFreq_SmallBasket")
user_profiles['strata'] = user_profiles['freq_cat'].astype(str) + "_" + user_profiles['vol_cat'].astype(str)

print("Personas distribution :\n", user_profiles['strata'].value_counts(normalize=True))

# **STEP 3**: STRATIFY SPLIT (80/20)

In [ ]:
# We ONLY split the user_ids, based on the ‘strata’ column.
X_train_users, X_test_users = train_test_split(
    user_profiles['user_id'],
    test_size=0.20,
    random_state=42,
    stratify=user_profiles['strata']
)

train_user_list = X_train_users.tolist()
test_user_list = X_test_users.tolist()

# **STEP 4**: FINAL DATASETS CREATION

In [ ]:
df_train = df[df['user_id'].isin(train_user_list)].copy()
df_test = df[df['user_id'].isin(test_user_list)].copy()

print(f"Shape Train: {df_train.shape} (Users NB: {len(train_user_list)})")
print(f"Shape Test: {df_test.shape} (Users NB: {len(test_user_list)})")

# Checking if the values are close between Train and Test for average days
print("\nAverage days without orders (Train) :", df_train.groupby('user_id')['days_since_prior_order'].mean().mean())
print("Average days without orders (Test)  :", df_test.groupby('user_id')['days_since_prior_order'].mean().mean())

# **STEP 5**: LEARNING PHASE (TRAIN SET)

In [ ]:
# Keep only products that have a certain sales volume
top_products = df_train['product_name'].value_counts()
top_products = top_products[top_products > 1000].index

df_train_filtered = df_train[df_train['product_name'].isin(top_products)]
df_train_transactions = df_train_filtered.groupby('order_id')['product_name'].apply(list).values.tolist()

In [ ]:
from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder
import pandas as pd

# te_ary maxtrix (Sparse CSR format)
te = TransactionEncoder()
te_ary = te.fit(df_train_transactions).transform(df_train_transactions, sparse=True)

df_train_sparse = pd.DataFrame.sparse.from_spmatrix(te_ary, columns=te.columns_)
print(f"Dataset format : {df_train_sparse.shape}")

try:
    print("FP-Growth...")
    frequent_itemsets = fpgrowth(df_train_sparse, min_support=0.01, use_colnames=True)

    #Generate the rules
    rules_train = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.2)

    # Cleaning columns for better readability
    rules_train = rules_train[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
    print(f"Number of rules found : {len(rules_train)}")

except MemoryError:
    print("Explode again...")

In [ ]:
# Prepare Test Set transactions (List of sets for speed)
# We use df_test, which you created during stratified splitting.
df_test_transactions_list = df_test.groupby('order_id')['product_name'].apply(set).tolist()

# Validation function
def validate_rules(rules, test_txs):
    results = []
    for _, row in rules.iterrows():
        ant = row['antecedents']
        conq = row['consequents']

        # Number of transactions containing the antecedent
        n_ant = sum(1 for tx in test_txs if ant.issubset(tx))
        # Numbre of transactions containing both antecedents and consequent
        n_both = sum(1 for tx in test_txs if (ant | conq).issubset(tx))

        conf_test = n_both / n_ant if n_ant > 0 else 0

        results.append({
            'Rule': f"{set(ant)} -> {set(conq)}",
            'Train_Conf': row['confidence'],
            'Test_Conf': conf_test,
            'Lift': row['lift'],
            'Diff_Abs': abs(row['confidence'] - conf_test)
        })
    return pd.DataFrame(results)

# Compute validation
validation_results = validate_rules(rules_train, df_test_transactions_list)

# Display top 10 most stable rules
print(validation_results.sort_values('Diff_Abs').head(10))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_validation(validation_df):
    plt.figure(figsize=(10, 6))

    # Scatter plot of Train vs Test confidence
    sns.scatterplot(data=validation_df, x='Train_Conf', y='Test_Conf', size='Lift', hue='Lift', palette='viridis')

    # Diagonal line for perfect stability
    max_val = max(validation_df['Train_Conf'].max(), validation_df['Test_Conf'].max())
    plt.plot([0, max_val], [0, max_val], color='red', linestyle='--', label='Perfect Stability')

    plt.title('Validate Association Rules: Train vs Test Confidence')
    plt.xlabel('Confidence in the Train Set (Existing clients)')
    plt.ylabel('Confidence in the Test Set (New clients)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

plot_validation(validation_results)